# TUTORIAL: Real-time digital twin of a hydrogen-based annular combustor
 

We can now put everything we have learned together. Data assimilation using real experimental data

We develop a digital twin of the hydrogen-based annular combustor. We want to estimate the LOM parameters and states from raw experimental data from microphones.  Because the raw data may be biased, we need to model the bias in both, measurement data and model.


In [ ]:
from utils import set_working_directories, get_annular_data

data_folder, results_folder, figs_folder = set_working_directories('annular')
get_annular_data(data_folder) # Download the data if not already present

## 1. Load data 
Create the reference truth and the observations.



In [ ]:
from models.physical import Annular
from observations import Observations
import os

ER = 0.4875 + 0.025 # 0.4875 + np.arange(0, 4) * 0.025

t_start = Annular.t_transient
t_stop = t_start + Annular.t_CR * 15


truth = Observations(model = os.path.join(data_folder, 'ER_{}'.format(ER)),
                     t_start = t_start,
                     t_stop = t_stop,
                     Nt_obs = 35,
                     t_max = t_stop + Annular.t_transient,
                     add_noise = False
                     )


In [ ]:
Observations.plot_truth(truth, Nq=4, fig_width=12, window=0.025, f_max=10000)

## 2. Define the forecast model
This is the physical model which we will use to model the true data.
Here, we select the filter parameters and create ensemble

*The function ```create_ensemble``` consists of* 

```
alpha0_mean = dict()
for alpha, lims in alpha0.items():
    alpha0_mean[alpha] = 0.5 * (lims[0] + lims[1])

ensemble = Annular(**alpha0_mean)

filter_params = dict(m= 20, 
                     std_psi=0.3,
                     std_a=alpha0)

# Forecast model to initialise the ensemble after transient
state, t_ = ensemble.time_integrate(int(ensemble.t_CR / ensemble.dt))
ensemble.update_history(state[-1], reset=True)

ensemble.init_ensemble(**filter_params)
ensemble.close()
```

In [ ]:
from create import create_ensemble
import numpy as np

alpha0 = dict(nu=(-15., 30.),
              c2beta=(10, 50),
              kappa=(1.E-4, 2.E-4),
              epsilon=(5e-3, 8e-3),
              omega=(1090 * 2 * np.pi, 1095 * 2 * np.pi),
              theta_b=(0.5, 0.7),
              theta_e=(0.4, 0.6)
              )

forecast_params = dict(model=Annular, 
                       dt=truth.dt, 
                       m=20, 
                       std_psi=0.3, 
                       std_a=alpha0,
                       filter='rBA_EnKF',  # 'rBA_EnKF' 'EnKF' 'EnSRKF'
                       )

ensemble = create_ensemble(**forecast_params)

In [ ]:
# Visualize ensemble initialization
from plot_results import plot_ensemble
plot_ensemble(ensemble, reference_params={'kappa': 1e-4, 'omega': 2 * np.pi, 'epsilon': 1e-3})

## 4. Train an ESN to model the model bias
The procedure is the following

&emsp; i. Initialise ESN Bias class object
&emsp; ii. Create synthetic bias to use as training data 
&emsp; iii. Train the ESN
&emsp; iv. Create washout data

<br>

**4.1. Initialise the ESN**

In [ ]:
from bias_estimators import ESN

ensemble_ESN = ensemble.copy()

train_params = dict(bias_model=ESN, 
                    upsample=5,
                    N_units=50,
                    N_wash=10,
                    t_train=ensemble.t_transient / 3.,
                    t_test=ensemble.t_CR * 2,
                    t_val=ensemble.t_CR * 2,
                    # Training data generation options
                    augment_data=True,
                    biased_observations=True,
                    correlation_based_training=True,
                    seed_W=0,
                    N_folds=4,
                    L=20,
                    std_a=alpha0,
                    # Hyperparameter search ranges
                    rho_range=(0.5, 1.),
                    sigma_in_range=(np.log10(1e-5), np.log10(1e1)),
                    tikh_range=[1e-12, 1e-9]
                    )

ensemble_ESN.init_bias(**train_params)



**4.2 Create training data**

The details of the code inside ```create_bias_training_dataset()``` function is explained in the tutorial ```Class_Bias.ipynb```.

**4.3. Train the ESN**

The training convergence, hyperparameter optimization and testing results are saved in a pdf file in *figs_ESN* folder.

**4.4. Create washout data**

We retrieve from the raw data a ```N_wash``` number of observations to use for initialising the ESN, i.e., to perform the washout. 
The ESN initialization must be before the fist observation.

```
from create import create_washout
wash_t, wash_obs = create_washout(ensemble.bias, t=t_true, y_raw=y_raw)
```

In [ ]:
from utils import save_to_pickle_file, load_from_pickle_file
from create import create_bias_model

ESN_filename = 'ESN_case_annular_raw'
 
bias, wash_obs, wash_t = create_bias_model(ensemble,
                                            bias_params=train_params,
                                            training_dataset=truth,
                                            folder=results_folder,
                                            bias_filename=ESN_filename
                                            )


ensemble_BA = ensemble.copy()
ensemble_BA.bias = bias.copy()

## 5. Apply data assimilation
We now have all the ingredients to start our data assimilation algorithm.

In [ ]:
from data_assimilation import dataAssimilation

out = []
DA_kwargs = dict(y_obs=truth['y_obs'].copy(), t_obs=truth['t_obs'].copy(), std_obs=0.05, 
                 wash_obs=wash_obs, wash_t=wash_t)

Nt_extra = int(ensemble.t_CR / ensemble.dt) * 10
    
ks = [0, 5.] 

for kk in ks:
    ens = ensemble_BA.copy()
    ens.regularization_factor = kk    
    filter_ens = dataAssimilation(ens, **DA_kwargs.copy(), Nt_extra=Nt_extra)
    out.append(filter_ens.copy())

In [ ]:
from plot_results import *

print_parameter_results(out)
plot_states_PDF(out, truth, nbins=20, window=(truth['t_obs'][-1], truth['t_obs'][-1] + ensemble.t_CR * 5))
plot_RMS_pdf(out, truth, nbins=20)

for ens in out:
    plot_timeseries(ens, truth, plot_bias=True, dims=[0,1])
    plot_parameters(ens, truth)
    plot_covariance(ens)